### Retail E-Commerce Supply Chain: Production ETL Data Pipeline

#### Executive Summary & Business Context
E-commerce retail operations require resilient data processing infrastructure to analyze fluctuations in consumer purchasing behavior across regional supply chains. In large enterprise retail networks like Walmart—where digital commerce accounts for over \$80 billion in annual revenue (~13% of total enterprise sales)—seasonal milestones, major promotional cycles, and public holiday events drive substantial variance in demand.

To empower supply chain planning and commercial inventory forecasting, this project establishes an automated, modular Extract, Transform, Load (ETL) pipeline. The pipeline integrates relational sales transactions with macroeconomic indicators, enforces data quality standards, calculates standardized monthly revenue performance benchmarks, and persists clean analytical datasets with automated file-system validation.

#### Data Dictionary

| Feature | Type | Business Definition |
| :--- | :--- | :--- |
| `index` | Integer | Unique transaction row identifier |
| `Store_ID` | Integer | Retail store operational unit identifier |
| `Date` | String / Date | Start date of the recorded sales week (`YYYY-MM-DD`) |
| `Dept` | Integer | Store operating department classification code |
| `Weekly_Sales` | Float | Gross weekly department sales volume (USD) |
| `IsHoliday` | Integer | Binary holiday indicator flag (`1` = Holiday week, `0` = Standard week) |
| `Temperature` | Float | Regional average temperature on observation date (°F) |
| `Fuel_Price` | Float | Regional consumer fuel price per gallon (USD) |
| `CPI` | Float | Prevailing Consumer Price Index metric |
| `Unemployment` | Float | Prevailing local civilian unemployment rate |
| `MarkDown1`–`4` | Float | Anonymized promotional markdowns applied during sales cycles |
| `Size` | Integer | Total physical store footprint area |
| `Type` | String | Categorical store classification segment |

#### Data Extraction

In [ ]:
import os
import pandas as pd

# Extract function is already implemented
def extract(store_data, extra_data):
    extra_df = pd.read_parquet(extra_data)
    merged_df = store_data.merge(extra_df, on = "index")
    return merged_df

# Call the extract() function and store it as the "merged_df" variable
merged_df = extract(grocery_sales, "extra_data.parquet")

#### Transformation & Data Cleansing
Implementing production-grade cleaning routines to sanitize raw merged records:
1. **Numerical Imputation:** `Weekly_Sales` exhibits significant skewness and is imputed using the statistical median, while macroeconomic indexes (`CPI`, `Unemployment`) are imputed via the sample mean.
2. **Temporal Feature Engineering:** Standardizing date strings into `datetime64` representations and isolating calendar `Month` values for seasonal cohort tracking.
3. **Outlier & Noise Filtering:** Isolating departments generating significant revenue by enforcing a minimum weekly sales threshold of `Weekly_Sales > 10,000`.
4. **Schema Pruning:** Dropping intermediate identifiers, store size dimensions, and auxiliary promotional variables to optimize downstream memory utilization.

In [ ]:
# Create the transform() function with one parameter: "raw_data"
def transform(raw_data):
    # Impute missing values in weekly sales with median, CPI wth mean, and Unemployment with mean
    raw_data.fillna(
        {
            'Weekly_Sales': raw_data['Weekly_Sales'].median(),
            'CPI': raw_data['CPI'].mean(),
            'Unemployment': raw_data['Unemployment'].mean()
        }, inplace = True
    )

    # Extract month data from date column
    raw_data['Date'] = pd.to_datetime(raw_data['Date'], format="%Y-%m-%d")
    raw_data['Month'] = raw_data['Date'].dt.month

    # Filter to include rows where weekly sales are over $10,000
    raw_data = raw_data.loc[raw_data['Weekly_Sales'] > 10000, :]

    # Drop unnecessary columns
    raw_data = raw_data.drop(['index', 'Date', 'Temperature', 
                              'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 
                              'Dept', 'Size', 'Type'], axis=1)

    return raw_data

In [ ]:
# Call the transform() function and pass the merged DataFrame
clean_data = transform(merged_df)

#### Analytics & Aggregation
Constructing business-facing analytics by computing average weekly sales broken down by calendar month. The aggregated output provides executive leadership with macro-level demand baselines across annual retail cycles.

In [ ]:
# Create the avg_weekly_sales_per_month function that takes in the cleaned data from the last step
def avg_weekly_sales_per_month(clean_data):
    # Subset Month and Weekly Sales for analysis
    data = clean_data[['Month', 'Weekly_Sales']]
    
    # Group by Month, compute the average of Weekly_Sales
    agg_data = data.groupby('Month')[['Weekly_Sales']].mean()
    
    # Rename the column to Avg_Sales
    agg_data = agg_data.rename(columns={'Weekly_Sales': 'Avg_Sales'})
    
    # Reset index
    agg_data = agg_data.reset_index()
    
    # Round the Avg_Sales column to two decimal places
    agg_data['Avg_Sales'] = agg_data['Avg_Sales'].round(2)
    
    return agg_data

In [ ]:
# Call the avg_weekly_sales_per_month() function and pass the cleaned DataFrame
agg_data = avg_weekly_sales_per_month(clean_data)

#### Data Loading
Persisting transformed tabular assets directly into decoupled CSV targets (`clean_data.csv` and `agg_data.csv`) without writing DataFrame index offsets, ensuring compatibility with analytical query engines.

In [ ]:
# Create the load() function that takes in the cleaned DataFrame and the aggregated one with the paths where they are going to be stored
def load(full_data, full_data_file_path, agg_data, agg_data_file_path):
    # Save the full transformed DataFrame to a CSV file
    full_data.to_csv(full_data_file_path, index=False)
    # Save the aggregated DataFrame to a CSV file
    agg_data.to_csv(agg_data_file_path, index=False)

In [ ]:
# Call the load() function and pass the cleaned and aggregated DataFrames with their paths
load(clean_data, "clean_data.csv", agg_data, "agg_data.csv")

#### Data Validation
Implementing runtime validation checks using `os.path.exists()` to verify file-system artifact creation, asserting that data persistence completed successfully before handoff to downstream visualization or modeling systems.

In [ ]:
# Create the validation() function with one parameter: file_path to check whether the previous function was correctly executed
def validation(file_path):
    # Check whether the file path exists using os.path.exists()
    if not os.path.exists(file_path):
        # Raise an Exception if the file does not exist
        raise Exception(f"Validation Failed: The file at '{file_path}' does not exist.")

In [ ]:
# Call the validation() function and pass first, the cleaned DataFrame path, and then the aggregated DataFrame path
validation("clean_data.csv")
validation("agg_data.csv")